# McDonald Valley Exercise

In this problem, we've designed a hypothetical field problem that parallels the model construction and calibration process much like a real field problem. This exercise is intended to give the student experience in building a flow model from GIS data sets and performing some hand calibration on that flow model.

This problem will be approached in 3 - 4 stages depending on time:

**Stage 1**: Review the included information and project objectives to design an effective approach to simulating the system. In this stage you may want to consider the pros and cons of various boundary conditions, discretization schemes, etc...

**Stage 2** Develop a model for the predevelopment condition based on the existing data that are provided to you

**Stage 3** Calibrate the model using "hand calibration" techniques. *Note* we will give you an example of how to update an existing boundary condition with FloPy

**Stage 4** Simulate the effects of the proposed stresses on the system using the calibrated model from stage 3.


### Stage 1: Problem formulation and existing data:
**Background**

McDonald Valley is an undeveloped, closed alluvial valley surrounded by low permeability crystalline bedrock (figures 1 and 2). The valley is dominated by scenic Lake Harbaugh located in the northwest corner of the valley (lots are available). Lake Harbaugh sits in a gentle depression that is bounded by Sand Ridge, which extends from the western edge of the valley to the northeastern corner. The south face of Sand Ridge is relatively steep. South of Sand Ridge, the valley slopes gently toward the southern boundary. The Straight River, which has its headwaters at the base of sand ridge just south of Lake Harbaugh, flows south and eventually leaves the valley along the southern boundary.

You have been given the task of examining the possible effects of groundwater development that has been proposed for McDonald Valley. The following development has been proposed:
   1. A 268,000 ft3/d well in the southern part of the valley at either site 1 or site 2 as shown in figure 2. The well would provide a water supply for Virginia City located to the south of McDonald Valley.
   2. A 67,000 ft3/d well is located in the northern part of the valley (site 3 in figure 2). This well would provide natural spring water for Reilly's Premium Beverages, Inc.

The county is concerned about the effect of this development. Specifically,the county is concerned that:

   1. Development will cause excessive water table declines in the northern half of the valley where a number of summer homes have very shallow water table wells that are used for domestic supply. The county has established the requirement that any development not cause more than a 2 foot decline in the water table anywhere in the northern half of the valley. Consultants for the Reilly Brewing Company contend that pumping from the Reilly well will have a negligible effect on the water table because the well will be placed below a clay layer that occurs across much of the northern part of the valley.
   2. McDonald Valley is the site of the Pollock's Ford National Historic Site and Recreation Area located 1000 feet south of the headwaters of the Straight River. Pollock's Ford is the site of an heroic fording of the Straight River during the battle for Sand Ridge during the American Revolution. Even though easier routes around the north side of the headwaters were discovered within minutes of Pollock's heroic crossing, the event nevertheless remains one of extreme historic importance. The county maintains a stream gage at Pollock's Ford and has decided that any potential development by Virginia City or Reilly's Brewery must not reduce the stream flow at that site by more than 20 percent. In addition, the county is requiring that development not lead to any induced infiltration from the Straight River anywhere along its course.
   3. The county is requiring stream capture analysis for the proposed well sites.



**The study area**

McDonald valley encompasses an area of about $2.4 * 10^8 ft^2$ and is shown in the figures below

![maps](../data/class_project/combined_map.png)


**Hydrogeology**

The valley contains unconsolidated valley fill alluvium. No hydraulic tests have been performed in the valley, but the sediments are similar to those in other valleys in the area which generally have horizontal hydraulic conductivities ranging between 10 feet/day and 500 feet/day. The sediments are predominantly medium to coarse grained sands and some gravels, however a low permeability lake clay has been observed in some bore holes in the valley. A hydrogeologic framework has already been constructed for the valley sediments, and hydraulic conductivity estimates and layering has been provided by the geologists. These datasets are in raster format.

**Hydrology**

The hydrologic system in McDonald Valley is in steady state with no significant seasonal or short term fluctuations in conditions.

McDonald Valley has a **mean annual precipitation of 36 inches per year** based on measurements made at Lake Harbaugh. Other stations in the area also report annual precipitation of 36 inches per year.

There is no surface water inflow into the valley. The Straight River originates in McDonald Valley. The headwaters of the river is located 9000 feet upstream from the southern boundary of the valley. The **river stage at the southern boundary of the valley is defined to be 0**, the datum to which all other head measurements in the valley are referenced. **The stream gradient is 0.0002, which corresponds to a stage of 1.8 feet at the headwaters**. The **river is approximately 100 feet wide over its entire length and has a depth of 1 foot or less** in most locations. Stream gages are located at the river's southern discharge point and at Pollock's Ford.

*Measured streamflow*
   - Southern Boundary: 884,494 $ft^3/d$
   - Pollock's Ford: 96,402 $ft^3/d$

*Lake Harbaugh information*

Lake Harbaugh is a dominant hydrologic feature in the valley. A previous study of Lake Harbaugh yielded the following information:
   - Stage: 11 ft
   - Lake evaporation: 27 inches per year
   - Precipitaiton: 36 inches per year
   - Estimated area of the lake is: $1.625 * 10^7 ft^2 $ : A lake delineation was done with GIS and that data is stored in a shapefile (*hint: you will need to get the area of the lake from it later*)

Lake Harbaugh is a closed lake with no surface water inflow or outflow. The morphology of the lake basin consists of a relatively steep sloping shore that levels out to a very uniform depth of 16 feet within 50 feet of the lake shore. The lake bottom is sandy and free of fine grained sediments over most of its bottom, especially near the shore. Very minor amounts of fine grained sediments occur in the very center of the lake basin.

**Water table elevations**

The previous field study also sampled 17 wells throughout McDonald Valley and recorded water levels. These water level elevations (at the water table) are shown below and have been stored in a shapefile too.

![Water_levels](../data/class_project/water_table_elevations.png)

### Problem formulation
**Study Questions**
Take a few minutes to think through these questions and come up with a conceptual model of McDonald Valley.

   1. Describe the elements of the hydrology of the system that you believe will be the most important in your analysis

   2. Describe how you would treat the boundary conditions and stresses

   3. Think about how you would incorporate the effects of the lake and river into this model? How will you simulate them and with which package(s)

   4. Write a water budget for the lake given the information that you have. Is the lake a net source or sink of water for the groundwater system? Can you make a quantitative estimate of the net rate of groundwater recharge or discharge due to the lake.

   5. What is the total amount of recharge to McDonald Valley and how is it distributed?

### Stage 2: Develop a model that represents pre-development conditions

For each step in the model development process shapefile and raster data that you'll need will be listed for the step. Examples and "recipies" from the previous notebooks can and should be used to help process data and build the MODFLOW model. Our first step here is importing the required python packages for this project.

In [1]:
import flopy
import numpy as np
import pandas as pd
import geopandas as gpd
from pathlib import Path

from flopy.discretization import StructuredGrid, VertexGrid
from flopy.utils.triangle import Triangle
from flopy.utils.voronoi import VoronoiGrid

from flopy.utils import GridIntersect, Raster

#### Setup a MODFLOW6 simulation and Groundwater Flow model object

Add a MODFLOW-6 IMS solver package (`flopy.mf6.ModflowIms()`)

Add a MODFLOW-6 TDIS time discretization package (`flopy.mf6.ModflowTdis()`) and add a single stress period

Finally add a MODFLOW-6 Groundwater flow model object to the simulation (`flopy.mf6.ModflowGwf()`)

*Hint: Look back at the intro to Flopy notebook to see how this is done if you get stuck*